In [21]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

app_train_raw = pd.read_csv('../data/raw/application_train.csv')
app_split = pd.read_csv('../data/processed/application_split.csv')

app_merged = pd.merge(app_train_raw, app_split, on='SK_ID_CURR', how='inner')
print(len(app_train_raw) == len(app_split)) #true

True


In [17]:
features = app_merged[['SK_ID_CURR', 'TARGET', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'DAYS_BIRTH', 'SPLIT']].copy()
features['AGE_YEARS'] = (-features['DAYS_BIRTH'] / 365).round().astype(int)
features = features[['SK_ID_CURR', 'TARGET', 'AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AGE_YEARS', 'SPLIT']]
features.head()

,SK_ID_CURR,TARGET,AMT_CREDIT,AMT_INCOME_TOTAL,AGE_YEARS,SPLIT
0,100002,1,406597.5,202500.0,26,train
1,100003,0,1293502.5,270000.0,46,train
2,100004,0,135000.0,67500.0,52,train
3,100006,0,312682.5,135000.0,52,test
4,100007,0,513000.0,121500.0,55,train


In [18]:
feature_columns = ['AMT_CREDIT', 'AMT_INCOME_TOTAL', 'AGE_YEARS']

train_mask = features['SPLIT'] == 'train'
val_mask = features['SPLIT'] == 'val'

X_train = features.loc[train_mask, feature_columns]
y_train = features.loc[train_mask, 'TARGET']

X_val = features.loc[val_mask, feature_columns]
y_val = features.loc[val_mask, 'TARGET']

In [20]:
scaler = StandardScaler().set_output(transform="pandas")

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [24]:
lr_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)

lr_model.fit(X_train_scaled, y_train)

y_prediction = lr_model.predict(X_val_scaled)
y_proba = lr_model.predict_proba(X_val_scaled)

print(X_train_scaled)
print(y_prediction)
print(y_proba)

        AMT_CREDIT  AMT_INCOME_TOTAL  AGE_YEARS
0        -0.478560          0.113401  -1.496963
1         1.724561          0.343424   0.173830
2        -1.153223         -0.346647   0.675068
4        -0.214250         -0.162628   0.925687
5        -0.270152         -0.239302   0.173830
...            ...               ...        ...
307498    1.770783          0.098066   0.675068
307501   -0.818993         -0.300642  -0.661567
307502   -0.929658         -0.254637   0.006750
307507   -0.818993         -0.331312   1.092766
307510    0.188167         -0.039948   0.173830

[184506 rows x 3 columns]
[0 1 0 ... 1 1 1]
[[0.60054403 0.39945597]
 [0.47451357 0.52548643]
 [0.55448614 0.44551386]
 ...
 [0.38359419 0.61640581]
 [0.4971407  0.5028593 ]
 [0.43085209 0.56914791]]


In [25]:
feature_names = X_train.columns.tolist()

coef_df = pd.DataFrame({
    'feature':     feature_names,
    'coefficient': lr_model.coef_[0],
}).sort_values('coefficient', key=abs, ascending=False)

print("Коэффициенты (coef_)")
print(coef_df.to_string(index=False))

print("Свободный член (intercept_)")
print(lr_model.intercept_[0])

print("Порядок классов (classes_)")
print(lr_model.classes_)

Коэффициенты (coef_)
         feature  coefficient
       AGE_YEARS    -0.286864
      AMT_CREDIT    -0.098421
AMT_INCOME_TOTAL     0.000479
Свободный член (intercept_)
-0.039322576919291925
Порядок классов (classes_)
[0 1]
